# Testing the `Summary Workflow` n8n Workflow

Test cases are loaded from `unit_tests/summary/*/test.json`.
Each file contains a pre-built `input` dict (already in the flat webhook
shape) and an `expectedOutput` with the six Thai staff-memo fields.

### What the webhook returns
```json
{
  "subject": "...", "objective": "...", "debt_cause": "...",
  "offer_suitability": "...", "request_information": "...", "summary": "..."
}
```
Because these fields are LLM-generated free text, exact-match comparison is
not meaningful. The run-all section shows actual vs. expected side-by-side
for human review.

### URL note
- Test URL (n8n editor open): `{base}/webhook-test/{path}`
- Production URL (workflow Active): `{base}/webhook/{path}`

In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests

## 1. Configuration

In [ ]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "515736a7-e9eb-4ea0-81cb-bfd4a4248a8b"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead

MEMO_KEYS = ["subject", "objective", "debt_cause",
             "offer_suitability", "request_information", "summary"]


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

## 2. Load test cases from JSON

In [ ]:
def _find_project_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "unit_tests").is_dir():
            return p
        p = p.parent
    raise RuntimeError("Could not find project root (directory containing 'unit_tests/')")


def load_unit_tests(agent_type: str) -> list[dict]:
    root = _find_project_root()
    test_dir = root / "unit_tests" / agent_type
    tests = []
    for tc_dir in sorted(test_dir.iterdir()):
        test_file = tc_dir / "test.json"
        if test_file.is_file():
            with open(test_file, encoding="utf-8") as f:
                tests.append(json.load(f))
    print(f"Loaded {len(tests)} test cases from {test_dir}")
    return tests


TEST_CASES = load_unit_tests("summary")
for tc in TEST_CASES:
    print(f"  {tc['testId']}: {tc['testDescription']}")

## 3. Webhook caller

In [ ]:
def call_summary(payload: dict, timeout: int = 60) -> dict:
    """POST the pre-built input dict to the Summary webhook and return parsed JSON."""
    url = get_webhook_url()
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

## 4. Run a single example

Inspect one test case and confirm connectivity before running the full suite.

In [ ]:
sample_tc = TEST_CASES[0]
print(f"Test: {sample_tc['testId']} — {sample_tc['testDescription']}")
print("\nInput payload (truncated):")
payload_str = json.dumps(sample_tc["input"], ensure_ascii=False, indent=2)
print(payload_str[:1500], "..." if len(payload_str) > 1500 else "")
print("\nExpected output:")
print(json.dumps(sample_tc["expectedOutput"], ensure_ascii=False, indent=2))

# Uncomment once N8N_BASE_URL points to a real, reachable instance:
# result = call_summary(sample_tc["input"])
# print("\nActual output:")
# print(json.dumps(result, ensure_ascii=False, indent=2))

## 5. Run all test cases and inspect actual vs. expected

Because the Summary Agent produces LLM-generated Thai prose, we show
actual and expected side-by-side for human review. A `responded` flag
marks whether the call succeeded.

In [ ]:
rows = []
for tc in TEST_CASES:
    expected = tc["expectedOutput"]
    try:
        result = call_summary(tc["input"])
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "responded": True,
            **{f"exp_{k}": (expected.get(k) or "")[:120] for k in MEMO_KEYS},
            **{f"act_{k}": (result.get(k) or "")[:120] for k in MEMO_KEYS},
            "error": None,
        })
    except requests.exceptions.RequestException as exc:
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "responded": False,
            **{f"exp_{k}": (expected.get(k) or "")[:120] for k in MEMO_KEYS},
            **{f"act_{k}": None for k in MEMO_KEYS},
            "error": str(exc),
        })

results_df = pd.DataFrame(rows)
print(f"Responded: {results_df['responded'].sum()}/{len(results_df)}")
results_df[["testId", "scenario", "messageMode", "responded",
            "exp_subject", "act_subject", "error"]]

## 6. Inspect a single test case in detail

In [ ]:
def show_detail(results_df: pd.DataFrame, test_id: str, memo_keys=MEMO_KEYS):
    row = results_df[results_df["testId"] == test_id]
    if row.empty:
        print(f"Test {test_id} not found.")
        return
    row = row.iloc[0]
    print(f"=== {test_id} | {row['scenario']} | {row['messageMode']} ===")
    for k in memo_keys:
        print(f"\n--- {k} ---")
        print(f"EXPECTED: {row[f'exp_{k}']}")
        print(f"  ACTUAL: {row[f'act_{k}']}")


# show_detail(results_df, "TC-0001")

## Notes

- This workflow is **Active**, so the production URL is
  `{base_url}/webhook/515736a7-e9eb-4ea0-81cb-bfd4a4248a8b`.
- The `input` in each test.json is already in the flat webhook shape
  (equivalent to what `prepare_input_to_summarise` produces on the `Start`
  sub-workflow path) — no extra transformation is needed.
- Output fields are LLM-generated Thai prose and are shown for human review
  rather than automated exact-match comparison.
- Add more test cases by running the unit-test generator; they are picked up
  automatically the next time this notebook runs.